In [14]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

In [17]:
scopus_df = pd.read_csv(r"abstracts_sample.csv", low_memory=False)

In [18]:
clean_scopus_df = scopus_df[['Abstract', 'Year']].copy()
clean_scopus_df = clean_scopus_df.dropna(subset =['Abstract'])

In [19]:
clean_scopus_df['Year'] = pd.to_numeric(clean_scopus_df['Year'], errors='coerce')
clean_scopus_df = clean_scopus_df.dropna(subset=['Year'])
clean_scopus_df['Year'] = clean_scopus_df['Year'].astype(int)
clean_scopus_df = clean_scopus_df.reset_index(drop=True)

In [21]:
clean_scopus_df.shape

(200, 2)

In [22]:
academic_stopwords_extended = [
    'important', 'improve', 'improved', 'improvement', 'improves', 'improving',
    'included', 'including', 'increase', 'increased', 'increasing', 'impact',
    'study', 'patients', 'results', 'conclusion', 'methods', 'background',
    'objective', 'objectives', 'aim', 'data', 'analysis', 'using', 'used',
    'especially', 'essential', 'established', 'estimate', 'estimated', 'estimation',
    'evaluate', 'evaluated', 'evaluation', 'evidence', 'experienced', 'expert', 'experts',
    'clinical', 'research', 'review', 'literature', 'article', 'articles', 'author', 'authors',
    'associated', 'compared', 'significant', 'significantly', 'higher', 'lower', 'additionally',
    'address', 'abstract', 'access', 'according', 'accurate', 'accurately','achieve', 'achieved',
    'achieving', 'acquired','addition', 'additional','additionally', 'american', 'analyses', 'analyze',
    'analyzed','assess', 'assessed', 'assessing', 'assessment','assessments', 'assist', 'assisted',
    'association', 'associations', 'attention', 'attenuation', 'available', 'average', 'baseline',
    'basis', 'behalf', 'benefit', 'best', 'better', 'bias', 'absence', 'absolute', 'acquisition',
    'activity', 'additive', 'adjusted', 'adjustment', 'admission', 'admitted', 'aged', 
    'agreement', 'aimed', 'aims', 'alternative', 'application', 'applications', 'applied',
    'approach', 'approaches', 'areas', 'artifacts', 'automated', 'automatic', 'automatically',
    'binary', 'built', 'burden', 'calculated', 'care', 'case', 'cases', 'categories', 
    'cause', 'center', 'centers', 'challenge', 'challenges', 'challenging', 'adults', 'applying',
    'characteristic', 'characteristics', 'characterization', 'china', 
    'classified', 'classify', 'clinically', 'body', 'category', 'change', 'changes',
    'achieves', 'characterized', 'collected', 'college', 'combination', 
    'combined', 'combining', 'commercial', 'common', 'comparable', 'compare', 'comparing'
     'copyright', 'elsevier', 'springer', 'wiley', 'wolters', 'kluwer', 'licence', 
    'license', 'llc', 'press', 'journal', 'periodicals', 'https', 'web', 'nature', 
    'science', 'oxford', 'wang', 'zhang', 'li', 'china', 'uk', 'european', 
    'international', 'national', 'worldwide', 'december', 'january', 'june', 
    'month', 'months', 'day', 'days', 'year', 'times', 'current', 'currently', 
    'future', 'recent', 'university', 'department', 'hospital', 'hospitals', 
    'society', 'et', 'paper', 'work', 'demonstrate', 'demonstrated', 'demonstrates', 
    'demonstrating', 'different', 'differences', 'furthermore', 'finally', 
    'specifically', 'statistically', 'subsequently', 'respectively', 'regarding', 
    'previously', 'particularly', 'overall'
    'adding', 'adult', 'advances', 'al', 'appropriate', 'art', 'benefits', 'chinese', 
    'comparing', 'comparison', 'comparisons', 'component', 'components', 'composite', 
    'composition', 'comprehensive', 'concordance', 'conditions', 'conducted', 'confidence', 
    'confirmed', 'consecutive', 'consensus', 'considered', 'consistency', 'consistent', 
    'consistently', 'construct', 'constructed', 'consuming', 'continuous', 'contrast', 
    'control', 'controls', 'conventional', 'core', 'correction', 'correlated', 'correlation', 
    'correlations', 'corresponding', 'cost', 'costs', 'count', 'created', 'criteria', 
    'critical', 'crucial', 'curves', 'database', 'dataset', 'datasets', 'death', 'deaths', 
    'decision', 'decisions', 'decreased', 'defined', 'degree', 'demographic', 'demographics', 
    'dependent', 'derivation', 'derived', 'descending', 'design', 'designed', 'despite', 
    'detect', 'detected', 'detecting', 'detection', 'determine', 'determined', 'develop', 
    'developed', 'developing', 'development', 'deviation', 'device', 'devices', 'diagnose', 
    'diagnosed', 'diagnoses', 'diagnosing', 'diagnosis', 'diagnostic', 'diameter', 'did', 
    'difference', 'digital', 'dimensional', 'discharge', 'distinct', 'distribution', 
    'diverse', 'divided', 'domain', 'dose', 'driven', 'drug', 'dual', 'duration', 'dynamic', 
    'early', 'effect', 'effective', 'effectively', 'effectiveness', 'effects', 'efficacy', 
    'efficiency', 'efficient', 'emerged', 'emergency', 'employed', 'enabled', 'enables', 
    'enabling', 'end', 'endpoint', 'enhance', 'enhanced', 'enhancement', 'enhances', 
    'enhancing', 'enrolled', 'error', 'establish', 'evaluating', 'event', 'events', 
    'examination', 'examined', 'excellent', 'exclusive', 'exercise', 'exhibited', 'existing', 
    'experience', 'experimental', 'experiments', 'explainable', 'explanations', 'explore', 
    'exposure', 'expression', 'extensive', 'external', 'externally', 'extract', 'extracted', 
    'extraction', 'extreme', 'facilitate', 'factor', 'factors', 'failure', 'false', 
    'feasible', 'feature', 'features', 'female', 'final', 'findings', 'flow', 'focus', 
    'fold', 'follow', 'followed', 'following', 'foundation', 'fraction', 'fractional', 
    'frame', 'frames', 'framework', 'free', 'frequency', 'fully', 'function', 'functional', 
    'fusion', 'gender', 'general', 'generate', 'generated', 'genes', 'genetic', 'given', 
    'global', 'goal', 'gold', 'good', 'grade', 'gradient', 'greater', 'group', 'groups', 
    'guide', 'guided', 'guidelines', 'having', 'hazard', 'health', 'healthcare', 'healthy', 
    'help', 'highest', 'highlights', 'highly', 'history', 'hospitalization', 'human', 
    'hybrid', 'identification', 'identified', 'identify', 'identifying', 'ii', 'image', 
    'images', 'imaging', 'implementation', 'implemented', 'importance', 'incidence', 
    'incident', 'include', 'inclusion', 'incorporated', 'incorporating', 'incremental', 
    'independent', 'independently', 'index', 'indicated', 'indicating', 'indices', 
    'individual', 'individualized', 'individuals', 'influence', 'information', 'initial', 
    'injury', 'input', 'insights', 'integrated', 'integrating', 'integration', 'intelligence', 
    'intensive', 'inter', 'interaction', 'intermediate', 'internal', 'interpretability', 
    'interpretable', 'interpretation', 'interval', 'intervention', 'interventions', 'intra', 
    'introduction', 'invasive', 'investigate', 'investigated', 'ir', 'iterative', 'key', 
    'kg', 'knowledge', 'known', 'label', 'laboratory', 'lack', 'large', 'larger', 'late', 
    'lead', 'leading', 'leads', 'left', 'length', 'level', 'levels', 'life', 'like', 
    'likelihood', 'likely', 'limitations', 'limited', 'linear', 'linked', 'local', 
    'localization', 'location', 'long', 'longitudinal', 'los', 'loss', 'low', 'lowest', 
    'main', 'major', 'making', 'male', 'management', 'manual', 'manually', 'map', 'maps', 
    'marker', 'markers', 'mass', 'matched', 'materials', 'maximum', 'mean', 'measured', 
    'measurement', 'measurements', 'measures', 'median', 'medical', 'medication', 'medicine', 
    'men', 'method', 'metrics', 'mg', 'mild', 'min', 'minimal', 'minimum', 'ml', 'mm', 
    'modeling', 'moderate', 'modified', 'monitoring', 'morbidity', 'morphological', 
    'morphology', 'motion', 'mr', 'mri', 'multi', 'multicenter', 'multimodal', 'multiple', 
    'multivariable', 'multivariate', 'nc', 'nearest', 'need', 'needed', 'negative', 'net', 
    'new', 'noise', 'nomogram', 'non', 'noninvasive', 'normal', 'novel', 'observational', 
    'observed', 'obstructive', 'obtained', 'occlusion', 'occurred', 'occurrence', 'odds', 
    'offering', 'offers', 'older', 'onset', 'open', 'operating', 'operator', 'optical', 
    'optimal', 'optimization', 'optimize', 'optimized', 'order', 'org', 'original', 
    'outcome', 'outcomes', 'outperformed', 'outperforming', 'pain', 'parameters', 
    'participants', 'pathways', 'pattern', 'patterns', 'peak', 'people', 'percent', 
    'percentage', 'percutaneous', 'perform', 'performed', 'performing', 'perfusion', 
    'period', 'peripheral', 'personalized', 'phase', 'phenotype', 'phenotypes', 'physical', 
    'physician', 'physicians', 'physiological', 'pipeline', 'planning', 'plasma', 'plots', 
    'point', 'points', 'poor', 'population', 'populations', 'positive', 'possible', 'post', 
    'postoperative', 'potential', 'potentially', 'power', 'practical', 'practice', 'pre', 
    'precise', 'precision', 'predict', 'predicted', 'predicting', 'predictions', 'predictive', 
    'predictor', 'predictors', 'prehospital', 'preoperative', 'presence', 'present', 
    'presented', 'presenting', 'presents', 'pressure', 'prevalence', 'prevention', 'previous', 
    'primary', 'prior', 'probability', 'procedural', 'procedure', 'procedures', 'process', 
    'processing', 'profile', 'profiles', 'prognosis', 'prognostic', 'program', 'progression', 
    'prolonged', 'promise', 'promising', 'proportion', 'proportional', 'propose', 'proposed', 
    'prospective', 'prospectively', 'protein', 'protocol', 'provide', 'provided', 'provides', 
    'providing', 'public', 'published', 'purpose', 'qualitative', 'quality', 'quantification', 
    'quantified', 'quantify', 'quantitative', 'radiation', 'random', 'randomized', 'randomly', 
    'range', 'ranging', 'rapid', 'rate', 'rates', 'ratio', 'reader', 'readers', 'real', 
    'recall', 'received', 'receiver', 'reclassification', 'recommendations', 'recommended', 
    'reconstructed', 'reconstruction', 'recorded', 'records', 'recurrent', 'reduce', 'reduced', 
    'reducing', 'reduction', 'reference', 'referred', 'regions', 'registration', 'registry', 
    'regression', 'related', 'relationship', 'relative', 'relevance', 'relevant', 'reliable', 
    'remain', 'remained', 'remains', 'remodeling', 'renal', 'report', 'reported', 'reporting', 
    'reports', 'require', 'required', 'requires', 'requiring', 'reserve', 'reserved', 
    'residual', 'resolution', 'response', 'rest', 'result', 'resulting', 'retinal', 
    'retrospective', 'retrospectively', 'revascularization', 'revealed', 'right', 'rights', 
    'risks', 'robust', 'role', 'routine', 'rule', 'safety', 'sample', 'samples', 'scale', 
    'scan', 'scans', 'scar', 'scores', 'scoring', 'screening', 'sd', 'second', 'secondary', 
    'sectional', 'segment', 'segmentation', 'segments', 'selected', 'selection', 'self', 
    'sensitivity', 'separate', 'serial', 'series', 'serve', 'set', 'sets', 'setting', 
    'settings', 'seven', 'severe', 'severity', 'sex', 'short', 'showed', 'shown', 'shows', 
    'shrinkage', 'signal', 'signals', 'significance', 'similar', 'simple', 'single', 'site', 
    'sites', 'size', 'sleep', 'small', 'smoking', 'social', 'software', 'sought', 'source', 
    'space', 'spatial', 'specific', 'specificity', 'split', 'sr', 'st', 'stable', 'stage', 
    'standard', 'standardized', 'state', 'statistic', 'statistical', 'status', 'step', 
    'strategies', 'strategy', 'stratification', 'stratified', 'stress', 'strong', 
    'structural', 'structured', 'studied', 'studies', 'subgroup', 'subgroups', 'subjective', 
    'subjects', 'subsequent', 'substantial', 'suggest', 'superior', 'supervised', 'support', 
    'supporting', 'surgery', 'surgical', 'survival', 'suspected', 'symptomatic', 'symptoms', 
    'syndrome', 'synthetic', 'systems', 'target', 'targeted', 'task', 'technique', 
    'techniques', 'technology', 'temporal', 'term', 'terms', 'test', 'tested', 'testing', 
    'tests', 'therapeutic', 'therapy', 'thickness', 'threshold', 'time', 'timely', 'tissue', 
    'tool', 'tools', 'traditional', 'train', 'trained', 'training', 'transfer', 'treated', 
    'treatment', 'tree', 'trend', 'trial', 'trials', 'type', 'types', 'unclear', 'undergoing', 
    'underwent', 'unique', 'unit', 'unstable', 'unsupervised', 'use', 'useful', 'utility', 
    'utilized', 'utilizing', 'validate', 'validated', 'valuable', 'value', 'values', 
    'variable', 'variables', 'various', 'vascular', 'vector', 'versus', 'vessel', 'vessels', 
    'visual', 'volume', 'volumes', 'vs', 'wall', 'wave', 'waveform', 'weighted', 'white', 
    'widely', 'women', 'world', 'yielded'
    'ability', 'able', 'added', 'adding', 'allow', 'allows', 'called', 
    'close', 'com', 'copyright', 'creative', 'creativecommons', 'date', 
    'did', 'does', 'gov', 'great', 'licensee', 'licenses', 'lippincott', 
    'make', 'mdpi', 'old', 'online', 'permissions', 'permits', 'publications', 
    'publishing', 'way', 'wide', 'wilkins', 'williams', 'wise', 'www', 
    'zero', 'april', 'august', 'july', 'march', 'september', 'october',
    'chen', 'liu', 'xu'
]
custom_stop_words = list(ENGLISH_STOP_WORDS) + academic_stopwords_extended

In [23]:
vectorizer = TfidfVectorizer(max_features=1000,
                             stop_words=custom_stop_words,
                             max_df = 0.25,
                             min_df = 6,
                             token_pattern=r'(?u)\b[a-zA-Z]{2,}\b')

In [24]:
tfidf_matrix = vectorizer.fit_transform(clean_scopus_df['Abstract'])

In [25]:
tfidf_matrix.shape

(200, 192)

In [27]:
past_df = clean_scopus_df[clean_scopus_df['Year'] <= 2022].copy()
present_df = clean_scopus_df[clean_scopus_df['Year'] > 2022].copy()

In [28]:
past_indices = past_df.index
present_indices = present_df.index

In [29]:
past_matrix = tfidf_matrix[past_indices]
present_matrix = tfidf_matrix[present_indices]

In [30]:
past_centroid = np.asarray(past_matrix.mean(axis=0))
present_centroid = np.asarray(present_matrix.mean(axis=0))

In [31]:
similarity_matrix = cosine_similarity(past_centroid, present_centroid)
result = similarity_matrix[0][0] * 100

In [32]:
print(f"{result:.2f}%")

84.85%
